# forecast_v0 — Deal Win Probability

Replacing the deterministic forecast (`amount × admin-set stage probability`) with a
learned, calibrated per-deal win probability.

**Population:** 12,963 opportunities from `dim_opportunity` — 11,051 closed (541 won, 4.9%), 1,912 open.

Docs: `field_analysis.md`, `framework.md`, `dataset_sample.md`

---

## ⚠️ Before running on Colab

`dim_cache.parquet` contains **real customer CRM data** — deal values, account
references, rep assignments. Uploading it to Colab puts internal data on
Google-hosted infrastructure, and Colab notebooks are shareable by link.
Clear this with data governance first, or run locally instead.

**Never paste `DEVREV_TOKEN` into a cell.** Notebooks persist output and get
committed. Cell 2 uses `getpass` if a live pull is needed.

## 1. Environment

In [ ]:
# Colab has pandas/sklearn/numpy preinstalled -- only pyarrow sometimes lags.
# Local: nothing to install if you already have .venv-ml.
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q pyarrow

import numpy as np, pandas as pd, sklearn
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

print(f"colab={IN_COLAB}  python={sys.version.split()[0]}  "
      f"sklearn={sklearn.__version__}  pandas={pd.__version__}")
print("reference run: python=3.9.6 sklearn=1.6.1 pandas=2.3.3")
print("NOTE: newer sklearn shifts HistGradientBoosting splits slightly -- "
      "expect PR-AUC to move a little vs the numbers in the docs.")

## 2. Data + source modules

The feature logic lives in `build_features.py`, the metrics in `evaluate.py`.
This notebook imports them rather than re-implementing — one definition of
the feature set, no drift between notebook and pipeline.

In [ ]:
import os
NEEDED = ["build_features.py", "evaluate.py", "dim_cache.parquet"]

if IN_COLAB and not all(os.path.exists(f) for f in NEEDED):
    from google.colab import files
    print("Upload: " + ", ".join(NEEDED))
    files.upload()

missing = [f for f in NEEDED if not os.path.exists(f)]
assert not missing, f"missing {missing} -- upload them or cd to the repo"

sys.path.insert(0, os.getcwd())
from build_features import (NO_AMOUNT_FEATURES, CONSERVATIVE_FEATURES,
                           LEAK_SAFE_FEATURES, POINT_IN_TIME_REQUIRED,
                           CATEGORICALS, parse_raw, clean)

print(f"model features ({len(NO_AMOUNT_FEATURES)}): {NO_AMOUNT_FEATURES}")
print(f"held for point-in-time: {POINT_IN_TIME_REQUIRED}")

In [ ]:
# Optional: refresh dim_cache.parquet from the live API.
# Skip unless the cache is stale -- getpass keeps the token out of saved output.
REFRESH = False

if REFRESH:
    from getpass import getpass
    from extract_pipeline import run_query
    token = getpass("DEVREV_TOKEN: ")   # never hardcode, never print
    DIM_SQL = "SELECT * FROM devrev.dim_opportunity WHERE is_deleted = false ORDER BY id"
    run_query(DIM_SQL, token, cache_path="dim_cache.parquet", use_cache=False)
    del token

## 3. Build features

In [ ]:
raw = pd.read_parquet("dim_cache.parquet")
train, score = clean(parse_raw(raw))

print(f"raw          {len(raw):>6,} deals")
print(f"train        {len(train):>6,} closed   {train.is_won.sum()} won "
      f"({train.is_won.mean():.1%})")
print(f"score        {len(score):>6,} open/in_progress")
print(f"\nstall_ratio median -- won {train.loc[train.is_won==1,'stall_ratio'].median():.3f} "
      f"| lost {train.loc[train.is_won==0,'stall_ratio'].median():.3f} "
      f"| open {score.stall_ratio.median():.3f}")

## 4. What the dataset looks like

Stratified sample — the table for the deck.

In [ ]:
SHOW = ["display_id"] + NO_AMOUNT_FEATURES + ["acv", "stage_name"]

def strat_sample(df, n, label):
    """Deterministic spread across display_id -- no random_state to explain."""
    d = df.sort_values("display_id")
    return d.iloc[:: max(1, len(d) // n)][:n].assign(outcome=label)

sample = pd.concat([
    strat_sample(train[train.is_won == 1], 8, "WON"),
    strat_sample(train[train.is_won == 0], 9, "LOST"),
    strat_sample(score, 8, "OPEN"),
])
sample[SHOW + ["outcome"]].reset_index(drop=True)

In [ ]:
# stall_ratio is the whole story: winners advance, losers park.
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(13, 4))

for lbl, m, c in [("won", train.is_won == 1, "tab:green"),
                  ("lost", train.is_won == 0, "tab:red")]:
    ax[0].hist(train.loc[m, "stall_ratio"], bins=25, density=True,
               alpha=0.55, label=lbl, color=c)
ax[0].hist(score.stall_ratio, bins=25, density=True, histtype="step",
           lw=2, label="open", color="tab:blue")
ax[0].set(xlabel="stall_ratio", ylabel="density",
          title="Share of cycle parked in current stage")
ax[0].legend()

bins = [0, 7, 14, 30, 60, 120, 10_000]
g = train.groupby(pd.cut(train.days_in_current_stage, bins))["is_won"].agg(["mean", "size"])
ax[1].bar(range(len(g)), g["mean"], color="tab:blue")
ax[1].set_xticks(range(len(g)))
ax[1].set_xticklabels([str(i) for i in g.index], rotation=30, ha="right")
ax[1].axhline(train.is_won.mean(), ls="--", c="k", label="base rate 4.9%")
ax[1].set(ylabel="win rate", title="Win rate by days in current stage")
ax[1].legend()
plt.tight_layout(); plt.show()

print("NOTE: open deals spike at stall_ratio 1.0 by construction -- a deal that")
print("never left stage 1 has days_in_current_stage == days_in_sales_cycle.")

## 5. Train + evaluate

Split by `created_date` — oldest 80% train, newest 20% test. **Never random**:
a random split leaks future outcomes into the past through account history.

Three feature sets scored identically so the cost of each exclusion stays visible.

In [ ]:
import warnings
warnings.filterwarnings("ignore")   # sigmoid calibrator overflows harmlessly at 4.9% positives

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from evaluate import encode, recall_at_top_k, reliability

df = train.sort_values("created_date").reset_index(drop=True)
cut = int(len(df) * 0.8)
tr, te = df.iloc[:cut], df.iloc[cut:]
ytr, yte = tr.is_won.to_numpy(), te.is_won.to_numpy()
print(f"train {len(tr):,} / {ytr.sum()} won   test {len(te):,} / {yte.sum()} won")

def new_hgb():
    # shallow on purpose: 492 training positives
    return HistGradientBoostingClassifier(
        max_depth=3, min_samples_leaf=50, max_iter=200, learning_rate=0.05,
        early_stopping=True, validation_fraction=0.15, random_state=0)

rows, fitted = [], {}
for label, feats in [("full", LEAK_SAFE_FEATURES),
                     ("conservative", CONSERVATIVE_FEATURES),
                     ("no_amount", NO_AMOUNT_FEATURES)]:
    xtr, xte, names = encode(tr, te, feats)

    lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=0.1)).fit(xtr, ytr)
    cal = CalibratedClassifierCV(new_hgb(), method="sigmoid", cv=3).fit(xtr, ytr)

    for mname, p in [("logreg", lr.predict_proba(xte)[:, 1]),
                     ("hgb_calibrated", cal.predict_proba(xte)[:, 1])]:
        rows.append({"model": f"{mname} [{label}]",
                     "PR-AUC": average_precision_score(yte, p),
                     "Brier": brier_score_loss(yte, p),
                     "ROC-AUC": roc_auc_score(yte, p),
                     "Recall@20%": recall_at_top_k(yte, p),
                     "sum_p/actual": p.sum() / max(yte.sum(), 1)})
    fitted[label] = (lr, cal, names, feats)

print(f"\nPR-AUC random baseline = {yte.mean():.4f}\n")
pd.DataFrame(rows).set_index("model").round(4)

**Reading it:** PR-AUC is the ranking metric (accuracy is useless at 4.9% —
"always lose" scores 95.1%). Brier is calibration, which matters most because
the product is a revenue number. `sum_p/actual` should sit near 1.0; above
that the model over-predicts and the roll-up will be optimistic.

In [ ]:
lr, cal, names, feats = fitted["no_amount"]
xtr, xte, _ = encode(tr, te, feats)
p = cal.predict_proba(xte)[:, 1]

rel = reliability(yte, p)
display(rel.round(4))

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot([0, 1], [0, 1], "k--", label="perfect")
ax[0].plot(rel.predicted, rel.actual, "o-", label="model")
for _, r in rel.iterrows():
    ax[0].annotate(f"n={r.n:.0f}", (r.predicted, r.actual), fontsize=7,
                   xytext=(4, -8), textcoords="offset points")
ax[0].set(xlabel="predicted", ylabel="actual", title="Reliability (bin counts shown)")
ax[0].legend()

coefs = pd.Series(lr[-1].coef_[0], index=names).sort_values(key=abs).tail(12)
ax[1].barh(coefs.index, coefs.values,
           color=["tab:red" if v < 0 else "tab:green" for v in coefs.values])
ax[1].axvline(0, c="k", lw=0.8)
ax[1].set(title="Drivers (logistic coefficients)", xlabel="← hurts    helps →")
plt.tight_layout(); plt.show()

print("Middle bins hold 5-17 deals -- that spread is noise as much as miscalibration.")
print("segment_unknown / source_unknown as drivers = the model partly learning")
print("CRM hygiene (blank fields lose more), not deal health. Real, but name it.")

## 6. Score the open pipeline

Refit on **all** closed deals, then score the 1,912 open ones. This is the
production output: a calibrated probability per deal, plus the revenue roll-up.

In [ ]:
x_all, x_open, _ = encode(train, score, NO_AMOUNT_FEATURES)
final = CalibratedClassifierCV(new_hgb(), method="sigmoid", cv=3).fit(x_all, train.is_won)

out = score[["display_id", "stage_name", "acv", "stall_ratio",
             "days_in_current_stage", "num_stakeholders", "segment",
             "opportunity_type"]].copy()
out["win_prob"] = final.predict_proba(x_open)[:, 1]
out["health"] = pd.cut(out.win_prob, [-0.01, 0.10, 0.30, 1.0],
                       labels=["Red", "Yellow", "Green"])

print(f"expected wins  {out.win_prob.sum():.0f} of {len(out):,} open deals")
print(f"expected revenue  ${(out.win_prob * out.acv).sum()/1e6:.1f}M")
print(f"  ^ amount is real on only {(score.acv_missing==0).mean():.0%} of open deals --")
print("    present as a RANGE with coverage stated, never a point estimate\n")
display(out.health.value_counts().rename("deals"))

In [ ]:
# The actionable slice: big money, low probability, long stalled.
risk = out[(out.acv > 50_000) & (out.win_prob < 0.10)].nlargest(15, "days_in_current_stage")
print("AT-RISK PIPELINE -- >$50k, <10% win prob, longest stalled\n")
display(risk.round(3).reset_index(drop=True))

print(f"\n${risk.acv.sum()/1e6:.1f}M sitting in these {len(risk)} deals. Today's forecast")
print("books every one of them at stage-constant x amount.")

## 7. Limits — read before quoting any number above

**Metrics are optimistic.** Training rows are closed deals in their *final*
state; scoring rows are open deals *mid-flight*. `days_in_sales_cycle` means
final cycle length on a closed deal (corr 0.95 with `actual_close − created`)
but age-so-far on an open one — same column, two meanings. Sizing that gap
needs backtesting, which needs snapshots that don't exist yet.

**541 wins is the binding constraint**, not model choice. Deeper models will
not help; more labelled mid-flight examples will.

**Three features are blocked on the same gap** — `fact_opportunity`'s changelog
starts 2026-05-01 while deals go back to 2023-01:

| Blocked feature | Why it needs history |
|---|---|
| stage number | On closed deals stage *is* the label (8=won, 9=lost) |
| close-date push count | `target_close_date` is overwritten to actual on 99.3% of closed deals |
| champion-absent-after-N-days | Needs when the champion field was filled, not whether |

**Excluded on purpose:** raw `created_date` (cohort censoring — win-rate-of-closed
runs 94.6% in 2023Q2 to 2.5% in 2026Q2 purely because old cohorts have resolved;
adding it drops PR-AUC 0.55 → 0.34), amount (`log_acv` AUC 0.334 — bigger deals
win *less*, size proxying for new-vs-renewal), and 10 leaking fields.

**Next:** start daily `dim_opportunity` snapshots. That single job unblocks
point-in-time training, all three features above, honest backtesting, and the
incumbent stage-constant baseline that decides ship/no-ship.